# Live DPD loop — ADRV9026

One notebook from the PA operating point to a linearized PA, on TX1 into ORx1.

1. **Operating point.** `CONDITION_SOURCE = "lock"` finds it here: transmit the signal,
   lock the TX attenuation on the target compression (`find_operating_point`, the same
   search as `pa_operating_point.ipynb`), optionally back off, capture, and write one CSV
   row. `"csv"` reuses a saved row (`CONDITIONS_CSV`, `CONDITION_NAME`) instead.
2. **Loop.** `linearize` holds that condition (LO, TX attenuation, ORx gain; no AGC) and
   repeats: transmit `u`, capture `z`, align `z` to `u`, score it against `x`, and call
   the DPD step for the next `u`. Iteration 0 sends `x` (no DPD).
3. **DPD step.** `IlaStep` with the built-in `GMP`: indirect learning. Every capture is
   scaled by the iteration-0 (no DPD) output peak, the *anchor*, so the output target stays
   in one place. It fits a post-inverse from that `z` to the transmitted `u` by block least
   squares on the samples around the peak, then predistorts `x` for an output peak
   `TARGET_BACKOFF_DB` below the anchor: a rule set once, not a backoff that grows each pass.
   The peak guard adds backoff only if the DPD peak would be above `PEAK_LIMIT_DBM`
   (full scale = the original input peak = 10 dBm). If `MAX_MARGIN_DB` in total is not
   enough, the loop stops and that waveform is never sent.

What one DPD pass does: `docs/dpd_pass.md`. The whole notebook: `docs/linearize_notebook.md`.

## 1. Parameters

In [ ]:
from adrvtrx import TxChannel, RxChannel

PROFILE = "ADRV9025Init_StdUseCase98_LinkSharing.profile"

TX_CHANNEL  = TxChannel.TX1
ORX_CHANNEL = RxChannel.ORX1

# Where the operating point comes from: "lock" = find it here, "csv" = reuse a saved row.
CONDITION_SOURCE = "lock"
CONDITIONS_CSV = "captures/operating_sweep/TX1_conditions.csv"     # "csv" only
CONDITION_NAME = "TX1_0dB_2200MHz_100BW"                          # "csv" only

# Lock ("lock" only): signal, bandwidth (for ACLR) and LO.
SIGNAL_PATH = "new100MHz256QAM_CFRed.txt"
BW_MHZ      = 100
LO1_HZ      = 2_400_000_000

# What the search locks on: "papr" = PAPR compression on the peak window (e.g. 3.0 dB),
# "gain" = gain compression at the signal peaks (e.g. 4.0 dB with COMP_TOL_DB = 0.2).
LOCK_ON               = "papr"
TARGET_COMPRESSION_DB = 3.0
COMP_TOL_DB           = 0.1
WINDOW_MS             = 0.1
MIN_TOP_SLOPE         = None     # clip guard, e.g. 0.08: back off before the PA output goes flat

# Safe start (more attenuation = less power). The floor is the hottest attenuation allowed.
TX_ATTEN_START_DB = 20.0
TX_ATTEN_MIN_DB   = 9.0
COARSE_STEP_DB    = 5.0
FINE_STEP_DB      = 0.2

ORX_TARGET_DBFS = -1.0
ORX_TOL_UP_DB   = 0.3
ORX_TOL_DOWN_DB = 0.6

BACKOFF_DB = 0                   # dB of attenuation above the lock; > 0 runs the ORx AGC again

# DPD loop
N_ITER = 4                       # captures, iteration 0 = no DPD
K, MEMORY, CROSS = 5, 5, 2       # GMP nonlinear order, memory depth, cross-term memory
N_TRAIN = 8192                   # post-inverse training block, centred on the peak of z
TARGET_BACKOFF_DB = 0.15         # output peak target, dB below the iteration-0 (no DPD) output peak
PEAK_LIMIT_DBM = 9.9             # never transmit a DPD waveform with a higher peak (full scale = 10 dBm)
MAX_MARGIN_DB  = 4.0             # largest total backoff (target + guard); above it the loop stops

SAVE_DIR = "DPD_DUT/ila_gmp"
LABEL = "ila_gmp"
OUTPUT_OVERSAMPLE = 2            # capture two periods so one aligned period can be cropped

## 2. Imports, config, profile, signal

In [ ]:
import csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.waveform import full_scale, load_tab_iq

from adrvtrx.conditions import load_aligned_iq, load_conditions
from adrvtrx.operating_point import find_operating_point
from adrvtrx.gmp import GMP
from adrvtrx.dpd import IlaStep, iteration_table, peak_dbm
from adrvtrx.linearize import linearize
from adrvtrx.metrics import papr_db

cfg = load_config()
cfg.profile_name = PROFILE
info = read_profile(cfg.profile_path)
FS = info.orx_rate_hz
print("Profile:", cfg.profile_path.name)
print(f"TX  {info.tx_rate_khz/1000:.2f} MSPS, {info.tx_bits} bit (FS {full_scale(info.tx_bits)})")
print(f"ORx {info.orx_rate_khz/1000:.2f} MSPS, {info.rx_bits} bit (FS {full_scale(info.rx_bits)})")

if CONDITION_SOURCE == "lock":
    signal = load_tab_iq(SIGNAL_PATH)
    freq_hz = int(LO1_HZ)
    print(f"Signal: {Path(SIGNAL_PATH).name}  N={len(signal)}  PAPR {papr_db(signal):.2f} dB  "
          f"-> lock at {freq_hz/1e6:.0f} MHz")
elif CONDITION_SOURCE == "csv":
    csv_dir = Path(CONDITIONS_CSV).parent
    condition = next((r for r in load_conditions(CONDITIONS_CSV) if r.name == CONDITION_NAME), None)
    if condition is None:
        raise ValueError(f"{CONDITION_NAME} is not in {CONDITIONS_CSV}")
    x = load_aligned_iq(csv_dir / condition.in_file)
    freq_hz = int(condition.freq_hz)
    print(f"{condition.name}: atten {condition.tx_atten_db:.2f} dB  ORx gain {condition.orx_gain}  "
          f"NMSE without DPD {condition.nmse_db:.2f} dB  N={len(x)}")
else:
    raise ValueError(f'CONDITION_SOURCE must be "lock" or "csv", got {CONDITION_SOURCE!r}')
cfg.lo.lo1_hz = freq_hz

## 3. Connect and program

In [ ]:
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()

start_atten = TX_ATTEN_START_DB if CONDITION_SOURCE == "lock" else condition.tx_atten_db
radio.set_tx_atten(TX_CHANNEL, start_atten)
lock = radio.retune_lo("LO1", freq_hz)

for k, v in verify_status(radio).items():
    print(f"{k}: {v}")
print(f"LO1 = {radio.get_lo('LO1')/1e6:.3f} MHz   lock 0x{lock:X}")
print(f"TX1 atten = {radio.get_tx_atten(TX_CHANNEL):.2f} dB")

## 4. Operating point

`"lock"`: TX starts at `TX_ATTEN_START_DB`, the ORx AGC levels the capture, and the search
lowers the attenuation until the `LOCK_ON` compression is within `COMP_TOL_DB` of the
target (the clip guard backs off if the AM/AM top slope drops below `MIN_TOP_SLOPE`).
With `BACKOFF_DB > 0` the attenuation goes up by that much and the AGC runs again. One
capture is aligned and scored, and the reference, the capture and the CSV row go to
`SAVE_DIR`. TX stops when the search ends, whatever happens.

`"csv"`: the saved row is used as it is.

In [ ]:
def show_search(rec):
    nan = float("nan")
    clip = "  CLIP" if rec.get("pa_clipped") else ""
    print(f"  atten {rec['atten_db']:5.2f} dB  gain {rec['orx_gain']:3d}  "
          f"peak {rec['peak_dbfs']:+5.1f} dBFS  railed {rec['railed']:<4d}  "
          f"PAPR comp {rec.get('papr_compression_db', nan):5.2f}  "
          f"gain comp {rec.get('gain_compression_db', nan):5.2f} dB  "
          f"slope {rec.get('top_slope', nan):5.3f}{clip}  {rec['action']}")

if CONDITION_SOURCE == "lock":
    op = find_operating_point(
        radio, TX_CHANNEL, ORX_CHANNEL, signal,
        tx_bits=info.tx_bits, rx_bits=info.rx_bits, fs=FS,
        freq_hz=freq_hz, bw_mhz=BW_MHZ, signal_name=Path(SIGNAL_PATH).name,
        lock_on=LOCK_ON, target_compression_db=TARGET_COMPRESSION_DB, comp_tol_db=COMP_TOL_DB,
        window_s=WINDOW_MS * 1e-3, min_top_slope=MIN_TOP_SLOPE,
        start_atten_db=TX_ATTEN_START_DB, atten_min_db=TX_ATTEN_MIN_DB,
        coarse_step_db=COARSE_STEP_DB, fine_step_db=FINE_STEP_DB,
        target_dbfs=ORX_TARGET_DBFS, tol_up_db=ORX_TOL_UP_DB, tol_down_db=ORX_TOL_DOWN_DB,
        backoff_db=BACKOFF_DB, oversample=OUTPUT_OVERSAMPLE, save_dir=SAVE_DIR,
        on_step=show_search,
    )
    res = op.search
    condition, x = op.condition, op.x
    print(f"\nconverged: {res.converged}   clip_limited: {res.clip_limited}   {res.reason}")
    print(f"lock {res.final_atten_db:.2f} dB   {res.lock_on} compression {res.compression_db:.2f} dB   "
          f"(PAPR {res.papr_compression_db:.2f}, gain {res.gain_compression_db:.2f} dB, "
          f"slope {res.top_slope:.3f})   row -> {op.csv_path}")
    if res.fatal:
        raise RuntimeError(f"operating point search failed: {res.reason}")
    if not res.converged:
        print("WARNING: the search did not converge; the loop runs where it stopped.")

c = condition
print(f"\n{c.name}: TX atten {c.tx_atten_db:.2f} dB (lock {c.locked_atten_db:.2f} dB)  "
      f"ORx gain {c.orx_gain}  {c.lock_on} compression at the lock {c.compression_db:.2f} dB")
print(f"without DPD: NMSE {c.nmse_db:.2f} dB   ACLR {c.aclr_lower_dbc:.2f} / {c.aclr_upper_dbc:.2f} dBc   "
      f"gain comp {c.gain_compression_db:.2f} dB   top slope {c.amam_top_slope:.3f}   "
      f"PA clipped {c.pa_clipped}   ORx peak {c.peak_dbfs:+.2f} dBFS")

## 5. The DPD step

`IlaStep` builds a fresh `GMP(K, MEMORY, CROSS)` on every pass. Inputs: `x` (the original
input), `u` (what was transmitted, 1.0 = DAC full scale) and `z` (the capture, aligned to
`u`). The first capture sets the anchor (its peak); later captures are divided by it, so
the output level each pass reaches is visible and the target does not slide. It returns
the next `u`, or `None` when no backoff up to `MAX_MARGIN_DB` keeps the peak at or below
`PEAK_LIMIT_DBM`.

In [ ]:
step = IlaStep(
    lambda: GMP(K, MEMORY, CROSS),
    n_train=N_TRAIN, target_backoff_db=TARGET_BACKOFF_DB, peak_limit_dbm=PEAK_LIMIT_DBM,
    max_margin_db=MAX_MARGIN_DB, verbose=True,
)
print(f"{GMP(K, MEMORY, CROSS)}: {GMP(K, MEMORY, CROSS).n_coeffs} coefficients, "
      f"training block {N_TRAIN} samples, output target {TARGET_BACKOFF_DB} dB below "
      f"iteration 0, peak limit {PEAK_LIMIT_DBM} dBm, total backoff up to {MAX_MARGIN_DB} dB")

## 6. Run the loop

The condition is applied once (LO, attenuation, ORx gain). The loop stops after `N_ITER`
captures, when the ORx rails (the gain is not re-levelled), or when the step cannot meet
the peak limit. TX stops when the loop ends, whatever happens.

In [ ]:
def show(it, rec):
    print(f"it {it}: ACLR {rec.aclr_lower_dbc:7.2f} / {rec.aclr_upper_dbc:7.2f} dBc   "
          f"NMSE {rec.nmse_db:7.2f} dB   gain comp {rec.gain_compression_db:5.2f} dB   "
          f"PAPR exp {rec.papr_expansion_db:5.2f} dB   railed {rec.railed}   "
          f"tx clipped {rec.tx_clipped}")

result = linearize(
    radio, condition, x, step,
    tx=TX_CHANNEL, orx=ORX_CHANNEL, tx_bits=info.tx_bits, rx_bits=info.rx_bits, fs=FS,
    n_iter=N_ITER, save_dir=SAVE_DIR, label=LABEL, oversample=OUTPUT_OVERSAMPLE,
    on_iter=show,
)
print(f"stopped: {result.reason}" + (f"  ({step.reason})" if step.reason else ""))

## 7. Per-iteration table

One row per capture. `out pk` is the output peak relative to iteration 0 (it should sit
near `-TARGET_BACKOFF_DB` on every DPD iteration). `gain comp` and `PAPR comp` are from `x`
to the capture: the PA's compression at iteration 0, what is left after it. `PA PAPR` is
the PA's PAPR compression on what it was actually fed (`papr(u) - papr(z)`), next to the
DPD's `PAPR exp`. `target`, `guard`, `DPD peak` and `post-inv` come from the pass that made
that iteration's `u` (none at iteration 0, which sends `x` at 10 dBm). Also written to
`{SAVE_DIR}/{LABEL}_steps.csv`; the full DUT columns are in `{SAVE_DIR}/{LABEL}.csv`.

In [ ]:
table = iteration_table(result.records, step.history)
cols = [
    ("iteration", "it"), ("aclr_lower_dbc", "ACLR low"), ("aclr_upper_dbc", "ACLR up"),
    ("aclr_worst_dbc", "worst"), ("nmse_db", "NMSE"), ("output_peak_db", "out pk"),
    ("gain_compression_db", "gain comp"), ("papr_compression_db", "PAPR comp"),
    ("pa_papr_compression_db", "PA PAPR"), ("papr_expansion_db", "PAPR exp"),
    ("dpd_peak_dbm", "DPD dBm"), ("target_backoff_db", "target"), ("guard_db", "guard"),
    ("tx_clipped", "tx clip"), ("post_inverse_nmse_db", "post-inv"),
]
print("".join(f"{h:>10}" for _, h in cols))
for row in table:
    print("".join(f"{row[k]:>10.2f}" if isinstance(row[k], float) else f"{row[k]:>10}"
                  for k, _ in cols))
print("dB / dBc / dBm; out pk is relative to iteration 0")
if step.reason:
    print(f"\nnot sent: {step.reason}")

steps_csv = Path(SAVE_DIR) / f"{LABEL}_steps.csv"
with open(steps_csv, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(table[0]))
    writer.writeheader()
    writer.writerows(table)
print(f"-> {steps_csv}")

## 8. Per iteration: ACLR, NMSE, output level, compression and PAPR, DPD peak; spectrum

In [ ]:
it = np.array([r["iteration"] for r in table])
peaks = [peak_dbm(x) if r["iteration"] == 0 else r["dpd_peak_dbm"] for r in table]

fig, ax = plt.subplots(2, 3, figsize=(17, 8))
ax = ax.flatten()
ax[0].plot(it, [r["aclr_lower_dbc"] for r in table], "o-", label="lower")
ax[0].plot(it, [r["aclr_upper_dbc"] for r in table], "s-", label="upper")
for r in table:
    ax[0].annotate(f"{r['aclr_worst_dbc']:.1f}", (r["iteration"], r["aclr_worst_dbc"]),
                   textcoords="offset points", xytext=(4, 6), fontsize=8)
ax[0].set_ylabel("ACLR (dBc)"); ax[0].set_title("ACLR per iteration (lower is better)")
ax[1].plot(it, [r["nmse_db"] for r in table], "o-")
ax[1].set_ylabel("NMSE vs x (dB)"); ax[1].set_title("NMSE per iteration")
ax[2].plot(it, [r["output_peak_db"] for r in table], "o-", label="output peak")
ax[2].axhline(-TARGET_BACKOFF_DB, c="g", lw=1, label=f"target -{TARGET_BACKOFF_DB} dB")
ax[2].set_ylabel("dB vs iteration 0"); ax[2].set_title("output peak kept")
ax[3].plot(it, [r["gain_compression_db"] for r in table], "o-", label="gain compression, x -> z")
ax[3].plot(it, [r["papr_compression_db"] for r in table], "s-", label="PAPR compression, x -> z")
ax[3].set_ylabel("dB"); ax[3].set_title("compression left")
ax[4].plot(it[1:], [r["papr_expansion_db"] for r in table[1:]], "o-", label="DPD PAPR expansion")
ax[4].plot(it[1:], [r["pa_papr_compression_db"] for r in table[1:]], "s-",
           label="PA PAPR compression on u")
ax[4].axhline(table[0]["papr_compression_db"], c="0.5", ls="--", lw=1,
              label="PA PAPR compression, iteration 0")
ax[4].set_ylabel("dB"); ax[4].set_title("PAPR expansion vs the PA's compression")
ax[5].plot(it, peaks, "o-", label="transmitted peak")
ax[5].axhline(PEAK_LIMIT_DBM, c="r", lw=1, label=f"limit {PEAK_LIMIT_DBM} dBm")
ax[5].axhline(10.0, c="0.5", lw=1, ls="--", label="full scale 10 dBm")
for r, p in zip(table, peaks):
    if r["iteration"] > 0:
        ax[5].annotate(f"{r['target_backoff_db']:.2f} + {r['guard_db']:.2f} dB",
                       (r["iteration"], p), textcoords="offset points", xytext=(4, -12),
                       fontsize=8)
ax[5].set_ylabel("peak (dBm)"); ax[5].set_title("DPD peak vs the limit (backoff: target + guard)")
for a in ax:
    a.set_xlabel("iteration"); a.set_xticks(it); a.grid(True)
for k in (0, 2, 3, 4, 5):
    ax[k].legend(fontsize=8)
plt.tight_layout(); plt.show()

def psd_db(s):
    n = len(s)
    p = np.abs(np.fft.fftshift(np.fft.fft(s * np.hanning(n)))) ** 2
    f = np.fft.fftshift(np.fft.fftfreq(n, 1.0 / FS)) / 1e6
    return f, 10 * np.log10(p / p.max() + 1e-30)

bw = condition.bw_mhz
z0 = load_aligned_iq(Path(SAVE_DIR) / result.records[0].out_file)
fig, ax = plt.subplots(figsize=(13, 4.5))
for s, label in ((x, "input x"), (z0, "it 0, no DPD"), (result.z, f"it {len(table) - 1}, DPD")):
    f, p = psd_db(s)
    ax.plot(f, p, lw=0.6, label=label)
for lo_edge, hi_edge in ((-1.5 * bw, -0.5 * bw), (0.5 * bw, 1.5 * bw)):
    ax.axvspan(lo_edge, hi_edge, color="0.85", alpha=0.5)
last = table[-1]
ax.set_title(f"ACLR {table[0]['aclr_lower_dbc']:.1f} / {table[0]['aclr_upper_dbc']:.1f} dBc without DPD -> "
             f"{last['aclr_lower_dbc']:.1f} / {last['aclr_upper_dbc']:.1f} dBc after {len(table) - 1} DPD passes")
ax.set_xlim(-FS / 2e6, FS / 2e6); ax.set_ylim(-90, 3)
ax.set_xlabel("MHz"); ax.set_ylabel("dB rel. peak"); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

## Safe-state and disconnect

In [ ]:
radio.safe_state()
radio.disconnect()
print("TX safe, board disconnected")